# 4  Flagging fraud in benefit claims

**Goal:** see why accuracy misleads on rare events and choose a threshold that fits the investigation team. The team can review **50 claims a week**.

**Data:** 10,000 synthetic benefit claims over 20 weeks; 2% are fraudulent.

**The data used here:**
- `benefit_claims.csv`: [view as a table](https://github.com/u1i/ml-foundations-workshop/blob/main/data/benefit_claims.csv) (opens on GitHub) · [download](https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/benefit_claims.csv) (right-click, *Save link as*)

**How this works:** each step is one cell. Click the &#9654; button on the left of the cell and read what appears below it. Where a cell has a field on the right, choose an option and click &#9654; again. No coding is needed. The cells marked *Stretch* at the end are optional and for those who like to code.

All data is synthetic.

In [ ]:
#@title Setup  (Click &#9654; first)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, HTML
DATA = "https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/"
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
RED, INK, GREY, GREEN, BLUE = "#A12527", "#1d1d1b", "#6e6e6a", "#2e7d4f", "#1f5a8a"
_STYLE = {"info": ("#1f5a8a", "#eef4f9"), "see": ("#A12527", "#fbeeee"), "ok": ("#2e7d4f", "#e9f5ee"),
          "try": ("#9a6a00", "#fbf3dd"), "step": ("#1d1d1b", "#f6f5f2")}

def box(title, text="", kind="info"):
    c, bg = _STYLE[kind]
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;border-left:6px solid {c};background:{bg};'
                 f'color:#1d1d1b;padding:14px 20px;margin:12px 0;font-size:17px;line-height:1.5;max-width:900px">'
                 f'<div style="font-size:20px;font-weight:bold;color:{c};margin-bottom:4px">{title}</div>{text}</div>'))

def big(*items):
    """big((value, label), (value, label, colour), ...): large numbers in one row."""
    cells = "".join(f'<div style="display:inline-block;margin:6px 40px 6px 0;vertical-align:top"><div style="font-size:15px;color:#6e6e6a">{it[1]}</div>'
                    f'<div style="font-size:44px;font-weight:bold;color:{it[2] if len(it) > 2 else INK}">{it[0]}</div></div>' for it in items)
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;color:#1d1d1b">{cells}</div>'))

def answer(choice, options):
    """options: {option: (is_right, explanation)}; shows feedback for the chosen option."""
    if choice not in options:
        box("Make your choice", "Pick an option in the field on the right of this cell, then click &#9654; again.", "try")
        return False
    right, why = options[choice]
    box("Correct" if right else "Not quite", why, "ok" if right else "try")
    return right

def table(df, highlight=None, n=10):
    """Show rows with cells highlighted: highlight = {(row_label, column): colour}."""
    sub = df.head(n).astype(object).where(df.head(n).notna(), "(empty)")
    def colour(_):
        out = pd.DataFrame("", index=sub.index, columns=sub.columns)
        for (r, c), col in (highlight or {}).items():
            if r in out.index and c in out.columns:
                out.loc[r, c] = f"background-color:{col};font-weight:bold"
        return out
    display(sub.style.apply(colour, axis=None).set_table_styles([{"selector": "td,th", "props": "font-size:14px;padding:4px 10px"}]))
import ipywidgets as W
def fslider(desc, lo, hi, step, val, fmt=".2f"):
    return W.FloatSlider(value=val, min=lo, max=hi, step=step, description=desc, continuous_update=False, readout_format=fmt,
                         layout=W.Layout(width="700px"), style={"description_width": "200px"})
def islider(desc, lo, hi, val):
    return W.IntSlider(value=val, min=lo, max=hi, description=desc, continuous_update=False,
                       layout=W.Layout(width="700px"), style={"description_width": "200px"})
live = W.interact

def matrix(y_true, y_flag, labels=("honest", "fraud"), flag_word="flagged"):
    """Confusion matrix as four large boxes."""
    y_true, y_flag = np.asarray(y_true).astype(bool), np.asarray(y_flag).astype(bool)
    tp, fn = int((y_true & y_flag).sum()), int((y_true & ~y_flag).sum())
    fp, tn = int((~y_true & y_flag).sum()), int((~y_true & ~y_flag).sum())
    cell = lambda n, t, c, bg: (f'<td style="border:2px solid #1d1d1b;padding:14px 18px;background:{bg};width:260px">'
                                f'<div style="font-size:15px;color:{c};font-weight:bold">{t}</div><div style="font-size:38px;font-weight:bold;color:{c}">{n:,}</div></td>')
    display(HTML(f'<table style="font-family:Arial,Helvetica,sans-serif;border-collapse:collapse;color:#1d1d1b;margin:8px 0">'
                 f'<tr><td></td><th style="padding:6px;font-size:15px">Model: {flag_word}</th><th style="padding:6px;font-size:15px">Model: not {flag_word}</th></tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[1]}</th>{cell(tp, "Found", GREEN, "#e9f5ee")}{cell(fn, "Missed", RED, "#fbeeee")}</tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[0]}</th>{cell(fp, "Checked for nothing", RED, "#fbeeee")}{cell(tn, "Correctly left alone", GREEN, "#e9f5ee")}</tr></table>'))
    return tp, fn, fp, tn

def bars(values, title, highlight=None, fmt="{:.2f}", color=INK):
    """Horizontal bars with values; values is a dict or Series."""
    s = pd.Series(values)
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(s) + 1))
    ax.barh(s.index.astype(str), s.values, color=[RED if k == highlight else color for k in s.index])
    for i, v in enumerate(s.values): ax.text(v, i, " " + fmt.format(v), va="center", fontsize=12)
    ax.invert_yaxis(); ax.set_title(title); plt.show()


box("Ready", "Setup done. Continue with Step 1 below.", "ok")

## Step 1  The claims

In [ ]:
#@title Step 1  Load the claims
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
df = pd.read_csv(DATA + "benefit_claims.csv")
INPUTS = [c for c in df.columns if c not in ("claim_id", "week", "fraud")]
X = pd.get_dummies(df[INPUTS], dtype=int); y = df["fraud"]
tr = df["week"] <= 15
X_train, X_test, y_train, y_test = X[tr], X[~tr], y[tr], y[~tr]
WEEKS = df.loc[~tr, "week"].nunique()
big((f"{len(df):,}", "claims"), (int(y.sum()), "fraudulent", RED), (f"{y.mean():.0%}", "share of fraud", RED))
box("How we test", f"The model learns from weeks 1 to 15 and is tested on the {WEEKS} weeks after: {len(X_test):,} claims, as it would be used in the service.", "info")

## Step 2  Is 98% good?

In [ ]:
#@title Step 2  A model that never flags fraud  (choose, then click &#9654;)
question = "Choose..." #@param ["Choose...", "Yes, 98% is excellent", "No, it finds no fraud at all"]
never = np.zeros(len(y_test), bool)
big((f"{accuracy_score(y_test, never):.1%}", "accuracy"), (0, "fraud cases found", RED))
_ = answer(question, {
    "Yes, 98% is excellent": (False, "It is right for every honest claim and wrong for every fraudulent one. With 2% fraud, doing nothing already scores 98%."),
    "No, it finds no fraud at all": (True, "Exactly. For rare events, accuracy says almost nothing. We need to ask how much fraud is found and how much work it creates."),
})

## Step 3  A real model

In [ ]:
#@title Step 3  Train a model and read the confusion matrix
model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=42).fit(X_train, y_train)
score = model.predict_proba(X_test)[:, 1]
flag = score >= 0.5
matrix(y_test, flag)
big((f"{accuracy_score(y_test, flag):.1%}", "accuracy"), (int(flag.sum()), "claims flagged at threshold 0.5"))
box("Do you see it?", "At the usual threshold of 0.5 the model flags almost nothing: most fraud cases get low scores, because fraud is rare. "
    "Accuracy still looks high. The two red boxes are the errors that matter.", "see")

## Step 4  Choose the threshold for 50 claims a week

In [ ]:
#@title Step 4  Move the threshold  (then move the slider)
def show(threshold):
    f = score >= threshold
    tp, fn, fp, tn = matrix(y_test, f)
    week = f.sum() / WEEKS
    big((f"{week:.0f}", "flagged per week", GREEN if abs(week - 50) <= 6 else RED),
        (f"{precision_score(y_test, f, zero_division=0):.0%}", "precision: flagged claims that are fraud"),
        (f"{recall_score(y_test, f):.0%}", "recall: fraud that is found"))
    if abs(week - 50) <= 6:
        box("This fits the team", f"About 50 claims a week. The model finds {tp} of {tp + fn} fraud cases; {fn} are still missed. "
            "<b>Report that number with the model</b>, so everyone knows what it does not catch.", "ok")
    elif week > 56: box("Too many", "More claims than the team can review. Move the threshold up.", "try")
    else: box("Capacity left", "The team could review more. Move the threshold down to find more fraud.", "try")
_ = live(show, threshold=fslider("Threshold", 0.01, 0.5, 0.01, 0.2))

In [ ]:
#@title Step 5  Precision or recall?  (choose, then click &#9654;)
question = "Choose..." #@param ["Choose...", "Precision", "Recall"]
box("A claim that is fraudulent but not flagged costs public money. Which number tells you how often that happens?", "", "step")
_ = answer(question, {
    "Precision": (False, "Precision tells you how many flagged claims are really fraud: the investigators' wasted effort."),
    "Recall": (True, "Recall tells you how much of all fraud the model finds. The rest is missed and paid out."),
})

## Result

In [ ]:
#@title Result
box("Do you see it now?", "A 98% accurate model can be useless. What counts is how much fraud is found at the workload the team can handle. "
    "<b>Now the team gets about 50 claims a week, with the riskiest first, and the service knows how much fraud still slips through.</b>", "see")

---
## Stretch (optional, for those who code)

In [ ]:
# Stretch: train with class_weight='balanced' and compare recall at 50 claims a week
w = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced", random_state=42).fit(X_train, y_train)
s2 = w.predict_proba(X_test)[:, 1]
top = np.zeros(len(s2), bool); top[np.argsort(-s2)[:50 * WEEKS]] = True
recall_score(y_test, top)